# P21 — Prédire les collisions avec blessés à New York (NYC Open Data)

**Contexte métier.** Programme Vision Zero de la ville de New York : prioriser les intersections et les créneaux horaires où une collision fait le plus souvent des blessés, pour cibler les interventions (modifications de voirie, campagnes, patrouilles). Un modèle binaire transforme le registre des collisions en carte de risque opérationnelle.

**Objectif de l'analyse exploratoire.** Identifier la cible (au moins un blessé) et les régresseurs (arrondissement, heure, facteurs contributifs, types de véhicules).

**Données.** NYC Open Data « Motor Vehicle Collisions - Crashes » (Socrata), export des 250 000 collisions les plus récentes.

## 1. Téléchargement

Export Socrata de la ville de New York (tri par date décroissante, 250 000 lignes).

In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv?$order=crash_date%20DESC&$limit=250000"
if not os.path.exists("data/nyc_collisions.csv"):
    urllib.request.urlretrieve(URL, "data/nyc_collisions.csv")
print("OK", os.path.getsize("data/nyc_collisions.csv") // (1024 * 1024), "Mo")

## 2. Chargement

Lecture avec typage souple, extraction de l'heure, construction de la cible binaire « au moins un blessé (blessé ou tué) ».

In [ ]:
import pandas as pd

df = pd.read_csv("data/nyc_collisions.csv", low_memory=False)
print(df.shape, "lignes x colonnes")
manq = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("% manquant (top 10) :")
print(manq.head(10).to_string())

df["crash_dt"] = pd.to_datetime(df["crash_date"], errors="coerce")
df["heure"] = pd.to_numeric(df["crash_time"].str.split(":").str[0], errors="coerce")
df["blesses"] = (df["number_of_persons_injured"].fillna(0) > 0) | (df["number_of_persons_killed"].fillna(0) > 0)
print()
print("Période :", df["crash_dt"].min().date(), "->", df["crash_dt"].max().date())
print("Part des collisions avec au moins un blessé :", round(100 * df["blesses"].mean(), 1), "%")

## 3. Analyse

Taux de blessés par arrondissement, par heure et par facteur contributif. Les types de véhicules 3 à 5 sont renseignés pour moins de 8 % des collisions (collisions mono-véhicule ou non saisies).

In [ ]:
print("Par arrondissement :")
print(df.groupby("borough")["blesses"].mean().mul(100).round(1).sort_values(ascending=False).to_string())
print()
print("Par heure (0-23) :")
print(df.groupby("heure")["blesses"].mean().mul(100).round(1).to_string())
print()
print("Facteurs contributifs (véhicule 1) :")
print(df["contributing_factor_vehicle_1"].value_counts().head(8).to_string())
print()
taux = df.groupby("contributing_factor_vehicle_1")["blesses"].agg(["mean", "count"])
taux = taux[taux["count"] > 500].sort_values("mean", ascending=False)
print("Taux de blessés par facteur (au moins 500 collisions) :")
print(taux["mean"].mul(100).round(1).head(6).to_string())

## 4. Résultats observés (exécution du 16/10/2026)

- 250 000 collisions, 29 colonnes ; période du 16/08/2023 au 11/06/2026 (250 000 plus récentes)
- 43,5 % des collisions font au moins un blessé (blessé ou tué)
- par arrondissement : Bronx 43,7 %, Brooklyn 43,7 %, Queens 42,0 %, Manhattan 41,1 %, Staten Island 36,7 %
- par heure : creux 3 h (34,5 %), pic 19 h (49,2 %) ; 17-19 h systématiquement > 45 %
- facteurs déclarés : Unspecified 63 650 (25 %), inattention/distraction 60 641, non-priorité 16 838
- taux de blessés par facteur : erreur piéton/cycliste 83,5 %, perte de connaissance 81,0 %, non-priorité 73,8 %
- manquants : types de véhicules 3-5 > 91 %, rue transversale 70 %, véhicule 2 : 34 %
- 31,2 % des heures manquantes venaient du format d'heure (9:40 vs 09:40) : normaliser avant d'analyser

## 5. Variable cible et régresseurs

**Cible (classification binaire) : au moins un blessé** (`number_of_persons_injured` > 0 ou `number_of_persons_killed` > 0) — 43,5 % de positifs, problème équilibré, utilisable tel quel avec un modèle classique (régression logistique, arbre, gradient boosting).

**Régresseurs retenus :**
- `borough`, `heure` (pic 17-19 h), jour de semaine
- `contributing_factor_vehicle_1/2` (inattention, non-priorité, vitesse)
- types de véhicules impliqués (`vehicle_type_code1/2`), implication d'un piéton ou cycliste
- rue (`on_street_name`), lat/lon quand disponibles (53 % de manquants)

**Pièges :**
- 25 % des facteurs sont « Unspecified » : les traiter comme une modalité, pas comme un manquant
- la cible est construite depuis les colonnes de comptage : ne pas les inclure comme features
- facteurs contributifs saisis après coup par la police : biais de déclaration ; « Lost Consciousness » (81 % de blessés) est un facteur constaté, pas un prédicteur ex ante
- l'export $limit=250000 trie par date : les données très récentes sont incomplètes (délai de saisie)
- l'heure doit être normalisée (str.split(':')) : slice(0,2) casse les heures 0-9